# Notebook 06 — Real Data Calibration

Calibrate model parameters against reference data using the DataCalibrator class.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.simulation import FleetSimulation
from src.data_calibration import DataCalibrator, load_or_generate_reference_data

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

HOURS_PER_DAY = 24
print('Ready.')

## Load Reference Data

In [ ]:
ref_data = load_or_generate_reference_data()
print(f'Reference data: {len(ref_data)} days')
print(f'Mean MCR: {ref_data["mcr"].mean()*100:.1f}%')
print(f'Std MCR:  {ref_data["mcr"].std()*100:.2f}%')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ref_data['day'], ref_data['mcr'] * 100, color='#3498db', linewidth=0.8)
ax.set_xlabel('Day')
ax.set_ylabel('MCR (%)')
ax.set_title('Reference MCR (Synthetic, Based on Published Benchmarks)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Run Calibration

In [ ]:
calibrator = DataCalibrator(FleetSimulation)

print('Running Weibull parameter calibration (this may take several minutes)...')
calibrated_params = calibrator.calibrate_weibull_parameters(
    ref_data['mcr'].values,
    n_quick_sims=20,
)

print('\nCalibrated Parameters:')
for param, value in calibrated_params.items():
    print(f'  {param}: {value:.1f}')

## Generate Simulated MCR with Calibrated Parameters

In [ ]:
calib_subsystem = {
    'engine': {'eta': calibrated_params['engine_eta']},
    'avionics': {'eta': calibrated_params['avionics_eta']},
    'hydraulics': {'eta': calibrated_params['hydraulics_eta']},
    'airframe': {'eta': calibrated_params['airframe_eta']},
}

sim = FleetSimulation(fleet_size=12, subsystem_params=calib_subsystem)

# Run multiple replications and extract daily MCR
daily_mcrs = []
for i in range(20):
    result = sim.run(simulation_days=365, random_seed=100 + i)
    kpi_df = result['kpi_dataframe']
    # Convert hourly to daily
    kpi_df['day_int'] = (kpi_df['time'] / HOURS_PER_DAY).astype(int)
    daily = kpi_df.groupby('day_int')['mcr'].mean()
    daily_mcrs.append(daily.values[:365])

min_day_len = min(len(d) for d in daily_mcrs)
sim_daily = np.mean([d[:min_day_len] for d in daily_mcrs], axis=0)
print(f'Simulated mean daily MCR: {sim_daily.mean()*100:.1f}%')

## Goodness-of-Fit Test

In [ ]:
fit_result = calibrator.goodness_of_fit_test(sim_daily, ref_data['mcr'].values)

print('KS Test Results:')
print(f'  KS Statistic: {fit_result["ks_statistic"]:.4f}')
print(f'  p-value:      {fit_result["p_value"]:.4f}')
print(f'  {fit_result["interpretation"]}')

## 4-Panel Calibration Figure

In [ ]:
fig = calibrator.plot_calibration_results(sim_daily, ref_data['mcr'].values)
plt.show()

## Save Calibration Results

In [ ]:
calibrator.save_calibration_results(calibrated_params, fit_result)

## Before vs After Calibration

In [ ]:
defaults = {'engine_eta': 800, 'avionics_eta': 600,
            'hydraulics_eta': 1200, 'airframe_eta': 2000}

comparison = pd.DataFrame({
    'Parameter': list(defaults.keys()),
    'Default': list(defaults.values()),
    'Calibrated': [calibrated_params[k] for k in defaults.keys()],
})
comparison['% Change'] = (
    (comparison['Calibrated'] - comparison['Default'])
    / comparison['Default'] * 100
).round(1)

print('\nParameter Comparison: Default vs Calibrated')
print('=' * 60)
comparison